# 04. Temporal Household Behavior

## Mục tiêu

> Hiểu hành vi household theo thời gian: mức độ hoạt động, độ dài lịch sử và khoảng quay lại.

Notebook dùng `DAY` và `WEEK_NO` là ordinal time. Mọi kết luận hiện dựa trên DAY 1–711, chưa gắn được ngày lịch thực.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl

from propensity.eda import (
    profile_return_intervals,
    summarize_household_distributions,
    summarize_period_activity,
)
from propensity.eda.dunnhumby import DUNNHUMBY_TRANSACTION_PROFILE_CONFIG
from propensity.eda.visualization import plot_return_interval_distribution
from propensity.io import scan_csv
from propensity.schema.dunnhumby import TRANSACTION_SCHEMA_OVERRIDES

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

## 1. Load transaction data

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

transactions = scan_csv(
    project_root / "data" / "raw" / "Dunnhumby" / "transaction_data.csv",
    schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
)
config = DUNNHUMBY_TRANSACTION_PROFILE_CONFIG

## 2. Tạo temporal evidence tables

In [ ]:
daily_activity = summarize_period_activity(transactions, config.time_col, config)
weekly_activity = summarize_period_activity(
    transactions, config.period_col, config
)
household_distributions = summarize_household_distributions(
    transactions, config
)
return_intervals = profile_return_intervals(transactions, config)

## 3. Activity theo ngày và tuần

Hai bảng giữ số product lines, baskets, households, products, sales, quantity và discount theo từng period. Đây là nền tảng để phân tích continuity, trend và seasonality.

In [ ]:
display(daily_activity.select(pl.exclude("period")).describe())
display(weekly_activity.select(pl.exclude("period")).describe())

### Kết luận period activity

- `daily_activity` có 711 dòng và `weekly_activity` có 102 dòng, đúng với độ phủ DAY/WEEK đã kiểm tra.
- Mỗi period chứa các measure ở cùng grain, cho phép xác định giai đoạn volume thấp/cao trước khi phân tích trend hoặc seasonality.
- `total_discount` là độ lớn khoản giảm, được tạo bằng cách đổi dấu tổng ba discount columns; quy ước dấu vẫn cần xác nhận từ data dictionary.

## 4. Active periods và độ dài lịch sử household

`n_active_days` đếm số DAY distinct có mua hàng. `span_days` tính từ DAY mua đầu tiên đến DAY mua cuối cùng, bao gồm cả hai đầu.

In [ ]:
household_distributions.filter(
    pl.col("metric").is_in(["n_active_days", "span_days"])
)

### Kết luận active history

- Median household có 70 active days.
- Median `span_days` là 627; p25 = 600 và p90 = 688.
- Phần lớn household có lịch sử đủ dài cho observation window 12 tháng, nhưng độ dài lịch sử không đồng nghĩa hoạt động liên tục. Continuity cần đọc cùng `n_active_days` và purchase intervals.

## 5. Khoảng thời gian quay lại

Khoảng quay lại là chênh lệch `DAY` giữa hai active days distinct liên tiếp của cùng household. Nhiều baskets trong cùng ngày không tạo gap bằng 0.

In [ ]:
return_interval_figure = plot_return_interval_distribution(return_intervals)
return_interval_figure

### Kết luận khoảng quay lại

- Có 223.033 khoảng quay lại hợp lệ từ 2.500 households.
- 24,0% lượt quay lại sau 1 ngày; cumulative share đạt 52,1% trong 3 ngày, 77,9% trong 7 ngày và 90,7% trong 14 ngày.
- Median = 3 ngày, p90 = 14, p95 = 22, p99 = 61 và max = 666 ngày.
- Phân phối chưa xử lý right censoring tại DAY 711 và không phải xác suất churn.

## 6. Phần temporal tiếp theo

Các bước tiếp theo trong chính notebook này sẽ bổ sung bảng household-level gồm first purchase, last purchase và recency tại snapshot; sau đó mới phân tích continuity, trend và seasonality. Mọi feature phải chỉ dùng dữ liệu có `DAY <= snapshot` để tránh leakage.